In [0]:
spark.conf.set(
  "fs.azure.account.key.salesviewacc.dfs.core.windows.net",
  "<accesskey>"
)


In [0]:
df = spark.read.csv(
  "abfss://bronze@salesviewacc.dfs.core.windows.net/sales_view/customer",
  header=True,
  inferSchema=True
)

display(df)


In [0]:
import re
from pyspark.sql.functions import split, col, when

# snake_case
def to_snake_case(c):
    return re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower()

df = df.toDF(*[to_snake_case(c) for c in df.columns])

# name split
df = df.withColumn("first_name", split(col("name"), " ")[0]) \
       .withColumn("last_name", split(col("name"), " ")[1])

# domain
df = df.withColumn("domain", split(col("email _id"), "@")[1])

# gender
df = df.withColumn("gender",
        when(col("gender")=="Male","M")
       .when(col("gender")=="Female","F"))

# date split
df = df.withColumn("date", split(col("joining _date"), " ")[0]) \
       .withColumn("time", split(col("joining _date"), " ")[1])

# expenditure
df = df.withColumn("expenditure_status",
        when(col("spent") < 200, "MINIMUM")
       .otherwise("MAXIMUM"))


In [0]:
df = df.withColumn("spent", col("spent").cast("double"))

df = df.withColumn("expenditure_status",
        when(col("spent").isNull(), "UNKNOWN")
       .when(col("spent") < 200, "MINIMUM")
       .otherwise("MAXIMUM"))


In [0]:
df = df.withColumnRenamed("email _id", "email_id") \
       .withColumnRenamed("joining _date", "joining_date") \
       .withColumnRenamed("customer _id", "customer_id") \
       .withColumnRenamed("order _i_d", "order_id")

In [0]:
print(df.columns)


#write to silver

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .save("abfss://silver@salesviewacc.dfs.core.windows.net/sales_view/customer")
